In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import pandas as pd
import numpy as np
import re, json, time, pickle, random
from tqdm import tqdm
from nltk.stem.snowball import RussianStemmer
import bm25s, faiss
import torch
from transformers import AutoTokenizer, AutoModel

torch.set_num_threads(8)
tqdm.pandas()
random.seed(42)

D = '../dataset'
M = '../models'
ANSW = '../answer.csv'

ARTIFACTS = {
    'tokens_full':    f'{D}/benchmark_items_tokens.jsonl',
    'tokens_fields':  f'{D}/item_tokens_fields.jsonl',
    'tokens_char3':   f'{D}/item_title_char3.jsonl',
    'bm25_full':      f'{D}/bm25_index',
    'bm25_title':     f'{D}/bm25_index_title',
    'bm25_char3':     f'{D}/bm25_index_char3',
    'e5_emb':         f'{D}/benchmark_items_e5.npy',
    'model_v3':       f'{D}/lgbm_reranker_v3.pkl',
}

print("Артефакты:")
for k, p in ARTIFACTS.items():
    print(f"  {k:<15} {'✓' if os.path.exists(p) else '✗'} {p}")

C:\Users\googl\python_global_env\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Артефакты:
  tokens_full     ✓ ../dataset/benchmark_items_tokens.jsonl
  tokens_fields   ✓ ../dataset/item_tokens_fields.jsonl
  tokens_char3    ✓ ../dataset/item_title_char3.jsonl
  bm25_full       ✓ ../dataset/bm25_index
  bm25_title      ✓ ../dataset/bm25_index_title
  bm25_char3      ✓ ../dataset/bm25_index_char3
  e5_emb          ✓ ../dataset/benchmark_items_e5.npy
  model_v3        ✓ ../dataset/lgbm_reranker_v3.pkl


In [2]:
train = pd.read_parquet(f'{D}/train.parquet')
benchmark_items = pd.read_parquet(f'{D}/benchmark_items.parquet')
benchmark_queries = pd.read_parquet(f'{D}/benchmark_queries.parquet')

train['item_id'] = train['item_id'].astype(str)
benchmark_items['item_id'] = benchmark_items['item_id'].astype(str)
benchmark_queries['query_id'] = benchmark_queries['query_id'].astype(str)

def to_float(s):
    if pd.isna(s): return np.nan
    s = str(s).replace(',', '.').replace(' ', '')
    try: return float(s)
    except: return np.nan

for df in [train, benchmark_items]:
    df['item_price'] = df['item_price'].apply(to_float)
    df['item_rating'] = pd.to_numeric(df['item_rating'], errors='coerce').fillna(-1)
    df['item_rating_reviews_count'] = pd.to_numeric(
        df['item_rating_reviews_count'], errors='coerce').fillna(-1)

print("train:", train.shape, "benchmark_items:", benchmark_items.shape)

train: (497673, 19) benchmark_items: (189212, 14)


In [3]:
stemmer = RussianStemmer()
TOKEN_RE = re.compile(r'[а-яa-z0-9]+')
_STEM_CACHE = {}

def tokenize_stem(text):
    if not isinstance(text, str) or not text: return []
    text = text.lower().replace('ё', 'е')
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) <= 1: continue
        s = _STEM_CACHE.get(t)
        if s is None:
            s = stemmer.stem(t); _STEM_CACHE[t] = s
        out.append(s)
    return out

def char_3gram_tokens(text):
    if not isinstance(text, str) or not text: return []
    text = text.lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9]', '', text)
    if len(text) < 3: return [text] if text else []
    return [text[i:i+3] for i in range(len(text) - 2)]

def make_query_key(df):
    cols = ['search_query','search_location_id','search_is_delivery_search',
            'search_infm_params_text','search_category']
    return df[cols].fillna('__NA__').astype(str).agg('||'.join, axis=1)

train['query_key'] = make_query_key(train)
train = train.drop_duplicates(subset=['query_key','item_id']).reset_index(drop=True)

bench_item_ids = set(benchmark_items['item_id'])
valid_pairs = train[train['item_id'].isin(bench_item_ids)].copy()
valid_groups = valid_pairs.groupby('query_key').agg(
    search_query=('search_query','first'),
    search_infm=('search_infm_params_text','first'),
    search_cat=('search_category','first'),
    search_location=('search_location_id','first'),
    relevant_items=('item_id', lambda x: list(set(x))),
).reset_index()

def build_query_text(row):
    parts = [str(row['search_query'])]
    if not pd.isna(row['search_infm']): parts.append(str(row['search_infm']))
    return ' '.join(parts)

valid_groups['query_text_full'] = valid_groups.apply(build_query_text, axis=1)
valid_groups['query_tokens'] = [tokenize_stem(t) for t in valid_groups['query_text_full']]
valid_groups['query_title_char3'] = [char_3gram_tokens(t) for t in valid_groups['search_query']]
print(f"valid_groups: {len(valid_groups)}")

valid_groups: 26556


In [4]:
MODEL_PATH = f'{M}/multilingual-e5-small'
device = 'cpu'

tokenizer_e5 = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model_e5 = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True).to(device).eval()

@torch.inference_mode()
def encode_e5(texts, batch_size=128, max_length=128):
    """Mean-pooling + L2 для e5-small. Сортировка по длине — меньше padding."""
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    out_all = []
    for i in range(0, len(sorted_texts), batch_size):
        batch = sorted_texts[i:i+batch_size]
        enc = tokenizer_e5(batch, padding=True, truncation=True,
                           max_length=max_length, return_tensors='pt').to(device)
        out = model_e5(**enc)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        emb = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        out_all.append(emb.cpu().numpy())
    result = np.vstack(out_all)
    inv = np.argsort(order)
    return result[inv]

print("e5-small загружена")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6597.79it/s]

e5-small загружена


In [5]:
# --- 1. Токены полного текста (full) ---
if not os.path.exists(ARTIFACTS['tokens_full']):
    print("Создаём токены full...")
    titles = benchmark_items['item_title_raw'].fillna('').astype(str)
    descs = benchmark_items['item_description_raw'].fillna('').astype(str)
    params = benchmark_items['item_infm_params_text'].fillna('').astype(str)
    item_text = (titles + ' ' + titles + ' ' + titles + ' ' + descs + ' ' + params)
    benchmark_items['tokens'] = [tokenize_stem(t) for t in tqdm(item_text, desc="tok full")]
    with open(ARTIFACTS['tokens_full'], 'w', encoding='utf-8') as f:
        for iid, toks in zip(benchmark_items['item_id'], benchmark_items['tokens']):
            f.write(json.dumps({'item_id': iid, 'tokens': toks}, ensure_ascii=False) + '\n')

# --- 2. Токены трёх полей (title/desc/params) ---
if not os.path.exists(ARTIFACTS['tokens_fields']):
    print("Создаём токены полей...")
    titles_tokens = [tokenize_stem(t) for t in benchmark_items['item_title_raw'].fillna('').astype(str)]
    descs_tokens = [tokenize_stem(t)[:100] for t in benchmark_items['item_description_raw'].fillna('').astype(str)]
    params_tokens = [tokenize_stem(t)[:100] for t in benchmark_items['item_infm_params_text'].fillna('').astype(str)]
    with open(ARTIFACTS['tokens_fields'], 'w', encoding='utf-8') as f:
        for iid, tt, dt, pt in zip(benchmark_items['item_id'], titles_tokens, descs_tokens, params_tokens):
            f.write(json.dumps({'item_id': iid, 'title': tt, 'desc': dt, 'params': pt}, ensure_ascii=False) + '\n')
    benchmark_items['tokens_title'] = titles_tokens
    benchmark_items['tokens_desc'] = descs_tokens
    benchmark_items['tokens_params'] = params_tokens

# --- 3. Char-3-граммы заголовка ---
if not os.path.exists(ARTIFACTS['tokens_char3']):
    print("Создаём char-3-граммы...")
    char3_tokens = [char_3gram_tokens(t) for t in tqdm(benchmark_items['item_title_raw'].fillna('').astype(str), desc="char3")]
    with open(ARTIFACTS['tokens_char3'], 'w', encoding='utf-8') as f:
        for iid, toks in zip(benchmark_items['item_id'], char3_tokens):
            f.write(json.dumps({'item_id': iid, 'tokens': toks}, ensure_ascii=False) + '\n')

# --- 4. BM25 индекс full ---
if not os.path.exists(ARTIFACTS['bm25_full']):
    print("Строим BM25 full...")
    if 'tokens' not in benchmark_items.columns:
        benchmark_items['tokens'] = [json.loads(l)['tokens'] for l in open(ARTIFACTS['tokens_full'], encoding='utf-8')]
    r = bm25s.BM25(); r.index(benchmark_items['tokens'].tolist())
    r.save(ARTIFACTS['bm25_full'])
    print("OK bm25_full")

# --- 5. BM25 индекс title ---
if not os.path.exists(ARTIFACTS['bm25_title']):
    print("Строим BM25 title...")
    if 'tokens_title' not in benchmark_items.columns:
        fields = [json.loads(l) for l in open(ARTIFACTS['tokens_fields'], encoding='utf-8')]
        title_tokens = [f['title'] for f in fields]
    else:
        title_tokens = benchmark_items['tokens_title'].tolist()
    r = bm25s.BM25(); r.index(title_tokens)
    r.save(ARTIFACTS['bm25_title'])
    print("OK bm25_title")

# --- 6. BM25 индекс char3 ---
if not os.path.exists(ARTIFACTS['bm25_char3']):
    print("Строим BM25 char3...")
    char3_data = [json.loads(l)['tokens'] for l in open(ARTIFACTS['tokens_char3'], encoding='utf-8')]
    r = bm25s.BM25(); r.index(char3_data)
    r.save(ARTIFACTS['bm25_char3'])
    print("OK bm25_char3")

# --- 7. e5-эмбеддинги заголовков ---
if not os.path.exists(ARTIFACTS['e5_emb']):
    print("Кодируем 189k заголовков через e5 (~11 мин)...")
    texts = ['passage: ' + str(t).strip() for t in benchmark_items['item_title_raw'].fillna('')]
    emb = encode_e5(texts, batch_size=128, max_length=64)
    np.save(ARTIFACTS['e5_emb'], emb)
    print(f"OK e5_emb: {emb.shape}")

print("\n=== Все артефакты на месте ===")


=== Все артефакты на месте ===


In [6]:
# e5 embeddings
item_emb_e5 = np.load(ARTIFACTS['e5_emb']).astype(np.float32)
faiss_index = faiss.IndexFlatIP(item_emb_e5.shape[1])
faiss_index.add(item_emb_e5)
print(f"FAISS: {faiss_index.ntotal} × {item_emb_e5.shape[1]}")

# BM25 индексы
r_full = bm25s.BM25.load(ARTIFACTS['bm25_full'], load_corpus=False)
r_title = bm25s.BM25.load(ARTIFACTS['bm25_title'], load_corpus=False)
r_char3 = bm25s.BM25.load(ARTIFACTS['bm25_char3'], load_corpus=False)

# Токены полей
tokens_by_id = {}
with open(ARTIFACTS['tokens_fields'], 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line); tokens_by_id[obj['item_id']] = obj

item_char3_by_id = {}
with open(ARTIFACTS['tokens_char3'], 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line); item_char3_by_id[obj['item_id']] = set(obj['tokens'])

print("Всё загружено")

FAISS: 189212 × 384
Всё загружено


In [7]:
valid_groups['e5_text'] = ['query: ' + str(q).strip() for q in valid_groups['search_query']]
t0 = time.time()
q_emb_e5 = encode_e5(valid_groups['e5_text'].tolist(), batch_size=128, max_length=64)
print(f"e5 queries: {time.time()-t0:.1f} сек, {q_emb_e5.shape}")

e5 queries: 62.6 сек, (26556, 384)


In [8]:
K_BM25 = 1000
K_E5 = 200
BATCH = 128

def batch_bm25(retriever, q_tokens_list, k, name):
    all_ids = [None] * len(q_tokens_list)
    all_scores = [None] * len(q_tokens_list)
    for s in tqdm(range(0, len(q_tokens_list), BATCH), desc=f"BM25 {name}"):
        e = min(s + BATCH, len(q_tokens_list))
        bq = q_tokens_list[s:e]
        safe = [q if q else ['_empty_'] for q in bq]
        res, sc = retriever.retrieve(safe, k=k, show_progress=False)
        for j in range(len(bq)):
            all_ids[s+j] = res[j]; all_scores[s+j] = sc[j]
    return all_ids, all_scores

q_tokens_list = valid_groups['query_tokens'].tolist()
q_char3_list = valid_groups['query_title_char3'].tolist()

bm25_full_ids, bm25_full_scores = batch_bm25(r_full, q_tokens_list, K_BM25, "full")
bm25_title_ids, bm25_title_scores = batch_bm25(r_title, q_tokens_list, K_BM25, "title")
bm25_char3_ids, bm25_char3_scores = batch_bm25(r_char3, q_char3_list, K_BM25, "char3")

e5_scores, e5_idxs = faiss_index.search(q_emb_e5, K_E5)
print("Поиски готовы")

BM25 char3: 100%|██████████| 208/208 [01:40<00:00,  2.08it/s]


Поиски готовы


In [9]:
bi = benchmark_items.reset_index(drop=True)
item_ids_array = bi['item_id'].values
item_microcat_arr = bi['item_microcat_id'].values
item_loc_arr = bi['item_location_id'].values
item_rating_arr = bi['item_rating'].values
item_reviews_arr = bi['item_rating_reviews_count'].values
item_price_log_arr = np.log1p(bi['item_price'].fillna(0).clip(lower=0).values)
item_title_len_arr = bi['item_title_raw'].fillna('').astype(str).str.len().values
item_desc_len_arr = bi['item_description_raw'].fillna('').astype(str).str.len().values

item_tokens_title_list = [tokens_by_id.get(iid,{}).get('title',[]) for iid in item_ids_array]
item_tokens_desc_list = [tokens_by_id.get(iid,{}).get('desc',[]) for iid in item_ids_array]
item_tokens_params_list = [tokens_by_id.get(iid,{}).get('params',[]) for iid in item_ids_array]
item_char3_sets = [item_char3_by_id.get(iid, set()) for iid in item_ids_array]

item_set_title = [set(t) for t in item_tokens_title_list]
item_set_desc = [set(t) for t in item_tokens_desc_list]
item_set_params = [set(t) for t in item_tokens_params_list]
item_set_all = [item_set_title[i] | item_set_desc[i] | item_set_params[i]
                for i in range(len(item_ids_array))]

rank_full = [{int(ip): r for r, ip in enumerate(a)} for a in bm25_full_ids]
rank_title = [{int(ip): r for r, ip in enumerate(a)} for a in bm25_title_ids]
rank_char3 = [{int(ip): r for r, ip in enumerate(a)} for a in bm25_char3_ids]
rank_e5 = [{int(ip): r for r, ip in enumerate(a)} for a in e5_idxs]
print("Предвычисления готовы")

Предвычисления готовы


In [10]:
feature_cols = [
    'bm25_full_score','bm25_full_rank',
    'bm25_title_score','bm25_title_rank',
    'bm25_char3_score','bm25_char3_rank',
    'e5_score','e5_rank','e5_cos',
    'overlap_recall','overlap_jaccard',
    'title_overlap_recall','desc_overlap_recall','params_overlap_recall',
    'char3_jaccard',
    'loc_match','cat_match','microcat_match',
    'item_rating','item_reviews','item_price_log',
    'item_title_len','item_desc_len',
]
print(f"Признаков: {len(feature_cols)}")

Признаков: 23


In [11]:
q_loc_arr = valid_groups['search_location'].astype(str).tolist()
q_cat_arr = valid_groups['search_cat'].astype(str).tolist()
q_char3_arr = valid_groups['query_title_char3'].tolist()

def compute_features_val(qp):
    pool = set(bm25_full_ids[qp].tolist()) | set(bm25_title_ids[qp].tolist()) | \
           set(bm25_char3_ids[qp].tolist()) | set(e5_idxs[qp].tolist())
    item_pos_list = list(pool)
    X = np.zeros((len(item_pos_list), len(feature_cols)), dtype=np.float32)

    qs = set(q_tokens_list[qp])
    q_char3 = set(q_char3_arr[qp])
    q_loc = q_loc_arr[qp]; q_cat = q_cat_arr[qp]
    q_emb = q_emb_e5[qp]
    qs_len = len(qs) if qs else 1
    qc3_len = len(q_char3) if q_char3 else 1

    lf, lt, lc3, le5 = rank_full[qp], rank_title[qp], rank_char3[qp], rank_e5[qp]
    sf_arr, st_arr = bm25_full_scores[qp], bm25_title_scores[qp]
    sc3_arr, se5_arr = bm25_char3_scores[qp], e5_scores[qp]

    for i, ip in enumerate(item_pos_list):
        at, ts, ds, ps = item_set_all[ip], item_set_title[ip], item_set_desc[ip], item_set_params[ip]
        rf = lf.get(ip, 9999); sf = float(sf_arr[rf]) if rf < 9999 else 0.0
        rt = lt.get(ip, 9999); st = float(st_arr[rt]) if rt < 9999 else 0.0
        rc3 = lc3.get(ip, 9999); sc3 = float(sc3_arr[rc3]) if rc3 < 9999 else 0.0
        re5 = le5.get(ip, 9999); se5 = float(se5_arr[re5]) if re5 < 9999 else 0.0
        cos_e5 = float(q_emb @ item_emb_e5[ip])

        if qs and at:
            inter = len(qs & at); ovr = inter / qs_len
            denom = qs_len + len(at) - inter
            ovj = inter / denom if denom > 0 else 0.0
        else: ovr = ovj = 0.0
        tor = len(qs & ts) / qs_len if qs else 0.0
        dor = len(qs & ds) / qs_len if qs else 0.0
        por = len(qs & ps) / qs_len if qs else 0.0
        ic3 = item_char3_sets[ip]
        if q_char3 and ic3:
            inter_c3 = len(q_char3 & ic3)
            denom_c3 = qc3_len + len(ic3) - inter_c3
            c3 = inter_c3 / denom_c3 if denom_c3 > 0 else 0.0
        else: c3 = 0.0

        X[i] = [sf, rf, st, rt, sc3, rc3, se5, re5, cos_e5,
                ovr, ovj, tor, dor, por, c3,
                int(q_loc == str(item_loc_arr[ip])),
                int(q_cat == str(item_microcat_arr[ip])),
                int(q_cat == str(item_microcat_arr[ip])),
                float(item_rating_arr[ip]), float(item_reviews_arr[ip]),
                float(item_price_log_arr[ip]),
                float(item_title_len_arr[ip]), float(item_desc_len_arr[ip])]
    return item_pos_list, X

In [12]:
if not os.path.exists(ARTIFACTS['model_v3']):
    print("Обучаем v3 с нуля...")
    import lightgbm as lgb
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import roc_auc_score

    MAX_POS = 10; N_NEG = 100
    item_to_pos = {iid: i for i, iid in enumerate(benchmark_items['item_id'])}

    pairs = []
    for pos in tqdm(range(len(valid_groups)), desc="Pairs"):
        relevant = set(valid_groups.iloc[pos]['relevant_items'])
        pool = set(bm25_full_ids[pos].tolist()) | set(bm25_title_ids[pos].tolist()) | \
               set(bm25_char3_ids[pos].tolist()) | set(e5_idxs[pos].tolist())
        pos_items = [item_to_pos[i] for i in relevant if i in item_to_pos]
        random.shuffle(pos_items); pos_items = pos_items[:MAX_POS]
        for ip in pos_items: pairs.append((pos, ip, 1))
        neg_pool = list(pool - set(pos_items)); random.shuffle(neg_pool)
        for ip in neg_pool[:N_NEG]: pairs.append((pos, ip, 0))

    pairs_df = pd.DataFrame(pairs, columns=['query_pos','item_pos','label']).reset_index(drop=True)
    print(f"Пар: {len(pairs_df)}, позитивов: {pairs_df['label'].sum()}")

    X_all = np.zeros((len(pairs_df), len(feature_cols)), dtype=np.float32)
    y_all = pairs_df['label'].values.astype(np.int8)
    idx_by_q = pairs_df.groupby('query_pos').indices

    for qp, row_idxs in tqdm(idx_by_q.items(), desc="Features"):
        item_pos_list, X_q = compute_features_val(qp)
        pos_to_idx = {ip: i for i, ip in enumerate(item_pos_list)}
        for ridx in row_idxs:
            ip = int(pairs_df.at[ridx, 'item_pos'])
            if ip in pos_to_idx:
                X_all[ridx] = X_q[pos_to_idx[ip]]

    query_text_by_pos = valid_groups['search_query'].to_dict()
    pairs_df['query_text_g'] = pairs_df['query_pos'].map(query_text_by_pos)
    gkf = GroupKFold(n_splits=5)
    lgb_params = dict(objective='binary', metric='auc', learning_rate=0.03,
                      num_leaves=63, min_child_samples=30, feature_fraction=0.8,
                      bagging_fraction=0.8, bagging_freq=5, lambda_l1=0.1, lambda_l2=0.1,
                      n_estimators=2000, n_jobs=8, random_state=42, verbose=-1)

    models = []
    for fold, (tr, va) in enumerate(gkf.split(pairs_df, groups=pairs_df['query_text_g'])):
        scale = (y_all[tr] == 0).sum() / max((y_all[tr] == 1).sum(), 1)
        m = lgb.LGBMClassifier(**lgb_params, scale_pos_weight=scale)
        m.fit(X_all[tr], y_all[tr], eval_set=[(X_all[va], y_all[va])], eval_metric='auc',
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
        auc = roc_auc_score(y_all[va], m.predict_proba(X_all[va])[:,1])
        print(f"Fold {fold+1}: AUC={auc:.4f}, best_iter={m.best_iteration_}")
        models.append(m)

    best_iter = int(np.mean([m.best_iteration_ for m in models]))
    scale = (y_all == 0).sum() / max((y_all == 1).sum(), 1)
    final_model = lgb.LGBMClassifier(
        **{k:v for k,v in lgb_params.items() if k!='n_estimators'},
        n_estimators=best_iter, scale_pos_weight=scale)
    final_model.fit(X_all, y_all)
    with open(ARTIFACTS['model_v3'], 'wb') as f:
        pickle.dump({'model': final_model, 'feature_cols': feature_cols}, f)
    print(f"OK model_v3, best_iter={best_iter}")
else:
    with open(ARTIFACTS['model_v3'], 'rb') as f:
        payload = pickle.load(f)
    final_model = payload['model']
    loaded_feature_cols = payload['feature_cols']
    assert loaded_feature_cols == feature_cols, "feature_cols не совпадают!"
    print("Загружена готовая модель v3")

Загружена готовая модель v3


In [13]:
benchmark_queries['q_tokens'] = [tokenize_stem(t) for t in benchmark_queries['search_query']]
benchmark_queries['q_char3'] = [char_3gram_tokens(t) for t in benchmark_queries['search_query']]
benchmark_queries['e5_text'] = ['query: ' + str(q).strip() for q in benchmark_queries['search_query']]

bq_emb_e5 = encode_e5(benchmark_queries['e5_text'].tolist(), batch_size=128, max_length=64)
bq_tokens = benchmark_queries['q_tokens'].tolist()
bq_char3 = benchmark_queries['q_char3'].tolist()

bq_full_ids, bq_full_scores = batch_bm25(r_full, bq_tokens, K_BM25, "bq_full")
bq_title_ids, bq_title_scores = batch_bm25(r_title, bq_tokens, K_BM25, "bq_title")
bq_char3_ids, bq_char3_scores = batch_bm25(r_char3, bq_char3, K_BM25, "bq_char3")
bq_e5_scores, bq_e5_idxs = faiss_index.search(bq_emb_e5, K_E5)

bq_rank_full = [{int(ip): r for r, ip in enumerate(a)} for a in bq_full_ids]
bq_rank_title = [{int(ip): r for r, ip in enumerate(a)} for a in bq_title_ids]
bq_rank_char3 = [{int(ip): r for r, ip in enumerate(a)} for a in bq_char3_ids]
bq_rank_e5 = [{int(ip): r for r, ip in enumerate(a)} for a in bq_e5_idxs]

bq_loc_arr = benchmark_queries['search_location_id'].astype(str).tolist()
bq_cat_arr = benchmark_queries['search_category'].astype(str).tolist()

def compute_features_bq(qp):
    pool = set(bq_full_ids[qp].tolist()) | set(bq_title_ids[qp].tolist()) | \
           set(bq_char3_ids[qp].tolist()) | set(bq_e5_idxs[qp].tolist())
    item_pos_list = list(pool)
    X = np.zeros((len(item_pos_list), len(feature_cols)), dtype=np.float32)

    qs = set(bq_tokens[qp]); q_char3 = set(bq_char3[qp])
    q_loc = bq_loc_arr[qp]; q_cat = bq_cat_arr[qp]; q_emb = bq_emb_e5[qp]
    qs_len = len(qs) if qs else 1; qc3_len = len(q_char3) if q_char3 else 1

    lf, lt = bq_rank_full[qp], bq_rank_title[qp]
    lc3, le5 = bq_rank_char3[qp], bq_rank_e5[qp]
    sf_arr, st_arr = bq_full_scores[qp], bq_title_scores[qp]
    sc3_arr, se5_arr = bq_char3_scores[qp], bq_e5_scores[qp]

    for i, ip in enumerate(item_pos_list):
        at, ts, ds, ps = item_set_all[ip], item_set_title[ip], item_set_desc[ip], item_set_params[ip]
        rf = lf.get(ip,9999); sf = float(sf_arr[rf]) if rf<9999 else 0.0
        rt = lt.get(ip,9999); st = float(st_arr[rt]) if rt<9999 else 0.0
        rc3 = lc3.get(ip,9999); sc3 = float(sc3_arr[rc3]) if rc3<9999 else 0.0
        re5 = le5.get(ip,9999); se5 = float(se5_arr[re5]) if re5<9999 else 0.0
        cos_e5 = float(q_emb @ item_emb_e5[ip])

        if qs and at:
            inter = len(qs & at); ovr = inter/qs_len
            denom = qs_len+len(at)-inter
            ovj = inter/denom if denom>0 else 0.0
        else: ovr = ovj = 0.0
        tor = len(qs & ts)/qs_len if qs else 0.0
        dor = len(qs & ds)/qs_len if qs else 0.0
        por = len(qs & ps)/qs_len if qs else 0.0
        ic3 = item_char3_sets[ip]
        if q_char3 and ic3:
            inter_c3 = len(q_char3 & ic3)
            denom_c3 = qc3_len+len(ic3)-inter_c3
            c3 = inter_c3/denom_c3 if denom_c3>0 else 0.0
        else: c3 = 0.0

        X[i] = [sf, rf, st, rt, sc3, rc3, se5, re5, cos_e5,
                ovr, ovj, tor, dor, por, c3,
                int(q_loc == str(item_loc_arr[ip])),
                int(q_cat == str(item_microcat_arr[ip])),
                int(q_cat == str(item_microcat_arr[ip])),
                float(item_rating_arr[ip]), float(item_reviews_arr[ip]),
                float(item_price_log_arr[ip]),
                float(item_title_len_arr[ip]), float(item_desc_len_arr[ip])]
    return item_pos_list, X

predictions = []
t0 = time.time()
for qp in tqdm(range(len(benchmark_queries)), desc="Inference v3"):
    item_pos_list, X_q = compute_features_bq(qp)
    scores = final_model.predict_proba(X_q)[:,1]
    top_idx = np.argsort(-scores)[:50]
    predictions.append([item_ids_array[item_pos_list[i]] for i in top_idx])
print(f"Инференс: {(time.time()-t0)/60:.1f} мин")

Inference v3: 100%|██████████| 2452/2452 [03:30<00:00, 11.65it/s]

Инференс: 3.5 мин


In [14]:
answer_df = pd.DataFrame({
    'query_id': benchmark_queries['query_id'].tolist(),
    'answer': [' '.join(p[:50]) for p in predictions],
})

assert (answer_df['query_id'].str.len() == 16).all()
assert answer_df['query_id'].nunique() == len(answer_df)
for ans in answer_df['answer']:
    ids = ans.split(' ')
    assert len(ids) == 50 and len(ids) == len(set(ids))
    for iid in ids: assert re.match(r'^[0-9a-f]{16}$', iid)

answer_df.to_csv(ANSW, index=False)
print(f"Сохранено в {ANSW}")

Сохранено в ../answer.csv
